### 219 REALHW 小红的最优分段量化

链接：[219 REALHW 小红的最优分段量化](https://www.nowcoder.com/practice/21ab2cd90b0f47a5bdbc6440055f3414?tpId=390&tqId=11745346&sourceUrl=%2Fexam%2Foj%2Fta%3FtpId%3D37)

![题目描述](desc.png)

In [ ]:
# 先使用暴力dp
# dp[k][i] 表示使用前i个，划分成k个区间，最小的总误差
# dp[k][i] = min(dp[k][i], dp[k - 1][j] + error(j + 1, i))
import sys
import math

data = sys.stdin.buffer.read().split()
it = iter(data)

N, K = int(next(it)), int(next(it))
a = [0 for _ in range(N + 2)]
s = [0 for _ in range(N + 2)]
for i in range(1, N + 1):
    a[i] = int(next(it))
    s[i] = s[i - 1] + a[i]

def mid(l, r):
    # return (l + r - 1) // 2 # 踩坑，对于 1-indexed 的序列，下中位数不该-1
    return (l + r) // 2

def error(l, r):
    global s, a
    m = mid(l, r)
    # m 及其左侧的误差
    el = (m - l + 1) * a[m] - (s[m] - s[l - 1])
    # m 右侧的误差
    er = (s[r] - s[m]) - (r - m) * a[m]
    e = el + er
    # e = (m - l + 1 - r + m) * a[m] + (s[r] + s[l - 1] - 2 * s[m])
    # e = (2 * m - l - r + 1) * a[m] + (s[r] + s[l - 1] - 2 * s[m])
    return e

# INF = int(1e9 + 10) # 踩坑，如果元素值都是-1e9，mid的值为1e9，那么最大误差在 N * 2e9 ≈ 1e13
INF = int(1e18)
dp = [[INF for _ in range(N + 2)] for _ in range(K + 2)]
dp[0][0] = 0


for k in range(K + 1):
    for i in range(N + 1):
        for j in range(i):
            dp[k][i] = min(dp[k][i], dp[k - 1][j] + error(j + 1, i))

print(dp[K][N])

"""
运行超时
您的程序未能在规定时间内运行结束，请检查是否循环有错或算法复杂度过大。
11/20 组用例通过
运行时间
4001ms
占用内存
4776KB
"""

踩坑：

1. INF = int(1e9 + 10) # 踩坑，如果元素值都是-1e9，mid的值为1e9，那么最大误差在 N * 2e9 ≈ 1e13
2. return (l + r - 1) // 2 # 踩坑，对于 1-indexed 的序列，下中位数不该-1

#### 怎么优化掉第三层循环呢？

先有个简单概念：

对于 i，k 固定时，使用暴力的方法：

dp[k - 1][j] + error(j + 1, i) 要遍历每一个 j ∈ [0, i - 1]

才能找到对于 i 而言的最优 j，记为：bestj = opt[i]

那么当 i + 1 时，此时的 bestj = opt[i + 1]

那么一个很朴素的猜想就是 i + 1 时的 opt[i + 1] ?>= opt[i]

这里可以通过四边形不等式验证（我不会）, 也可以通过打表粗略验证

由此，我们就可以将每个新的 i 时需要的 j ∈ [0, i - 1] 循环

优化成 j ∈ [opt[i - 1], i - 1]

In [ ]:
for k in range(K + 1):
    bestj = -1
    for i in range(N + 1):
        for j in range(max(0, bestj), i):
            if dp[k - 1][j] + error(j + 1, i) < dp[k][i]:
                dp[k][i] = dp[k - 1][j] + error(j + 1, i)
                bestj = j

"""
运行超时
您的程序未能在规定时间内运行结束，请检查是否循环有错或算法复杂度过大。
14/20 组用例通过
运行时间
4001ms
占用内存
6648KB
"""

显然这样的优化还不够，那么还能怎么优化呢？

先看这里的循环，对于每个 i 要找到 bestj 还是要从 opt[i - 1] 循环到 i - 1

那么，我们能不能直接优化 i ∈ [1, N] 这个循环呢？想了想似乎不太行

那就继续优化 bestj 的查找

结合二分的思想，我们对 i ∈ [1, N] 这个循环，不直接看 i = 1 , 而是先看 i = mid

那么先找到 mid 的 bestj , 结合前面的单调性, 就有 :
- i ∈ [1, mid] 时 opt[i] <= bestj_mid
- i ∈ [mid + 1, N] 时 bestj_mid <= opt[i]

如此不断的分治就能优化整个 i ∈ [1, N] 循环

In [ ]:
# 伪代码，分治处理

def solve(l, r, optl, optr):
    """
    l: 当前处理的 i 所要搜索的区间的左边界
    r: 当前处理的 i 所要搜索的区间的右边界
    optl: 当前处理的 i 所要搜索的区间的最优 j 的左边界
    optr: 当前处理的 i 所要搜索的区间的最优 j 的右边界
    
    curr: 当前 k 的 dp 数组
    prev: k - 1 的 dp 数组
    opt: 当前记录的每个i的最优j的位置
    """
    if l > r:
        return

    mid = (l + r) // 2
    # 先暴力找到 bestj_mid
    bestj_mid = -1
    best_cost = INF
    for j in range(optl, min(mid, optr) + 1):
        if best_cost > prev[j] + error(j + 1, mid):
            best_cost = prev[j] + error(j + 1, mid)
            bestj_mid = j
    
    curr[mid] = best_cost
    opt[mid] = bestj_mid
    # 再分治，处理 mid 左半部分和右半部分
    solve(l, mid - 1, optl, bestj_mid)
    solve(mid + 1, r, bestj_mid, optr)

In [ ]:
# 完整代码：

# 先使用暴力dp
# dp[k][i] 表示使用前i个，划分成k个区间，最小的总误差
# dp[k][i] = min(dp[k][i], dp[k - 1][j] + error(j + 1, i))
import sys
import math

data = sys.stdin.buffer.read().split()
it = iter(data)

N, K = int(next(it)), int(next(it))
a = [0 for _ in range(N + 2)]
s = [0 for _ in range(N + 2)]
for i in range(1, N + 1):
    a[i] = int(next(it))
    s[i] = s[i - 1] + a[i]

def mid(l, r):
    # return (l + r - 1) // 2 # 踩坑，对于 1-indexed 的序列，下中位数不该-1
    return (l + r) // 2

def error(l, r):
    global s, a
    m = mid(l, r)
    # m 及其左侧的误差
    el = (m - l + 1) * a[m] - (s[m] - s[l - 1])
    # m 右侧的误差
    er = (s[r] - s[m]) - (r - m) * a[m]
    e = el + er
    # e = (m - l + 1 - r + m) * a[m] + (s[r] + s[l - 1] - 2 * s[m])
    # e = (2 * m - l - r + 1) * a[m] + (s[r] + s[l - 1] - 2 * s[m])
    return e

# INF = int(1e9 + 10) # 踩坑，如果元素值都是-1e9，mid的值为1e9，那么最大误差在 N * 2e9 ≈ 1e13
INF = int(1e18)

curr = [INF for _ in range(N + 2)]
prev = [INF for _ in range(N + 2)]
opt = [-1 for _ in range(N + 2)]

def solve(l, r, optl, optr):
    """
    l: 当前处理的 i 所要搜索的区间的左边界
    r: 当前处理的 i 所要搜索的区间的右边界
    optl: 当前处理的 i 所要搜索的区间的最优 j 的左边界
    optr: 当前处理的 i 所要搜索的区间的最优 j 的右边界
    
    curr: 当前 k 的 dp 数组
    prev: k - 1 的 dp 数组
    opt: 当前记录的每个i的最优j的位置
    """
    global curr, opt, prev
    if l > r:
        return

    mid = (l + r) // 2
    # 先暴力找到 bestj_mid
    bestj_mid = -1
    best_cost = INF
    for j in range(optl, min(mid, optr) + 1):
        if best_cost > prev[j] + error(j + 1, mid):
            best_cost = prev[j] + error(j + 1, mid)
            bestj_mid = j
    
    curr[mid] = best_cost
    opt[mid] = bestj_mid
    # 再分治，处理 mid 左半部分和右半部分
    solve(l, mid - 1, optl, bestj_mid)
    solve(mid + 1, r, bestj_mid, optr)

prev[0] = 0
for k in range(1, K + 1):
    curr = [INF for _ in range(N + 2)]
    solve(1, N, 0, N - 1)
    prev = curr

print(curr[N])

"""
提交结果：答案正确 运行时间：1616ms 占用内存：6312KB 使用语言： Python 3 用例通过率：100.00%
"""